# Exercises XP Gold
Last Updated: March 7th, 2025

## 👩‍🏫 👩🏿‍🏫 What You’ll learn
Identify the limitations of traditional language models.
Understand the impact of attention mechanisms in transformer models.
Implement a basic Retrieval-Augmented Generation (RAG) system.
Analyze the role of BERT in RAG systems.
Compare and contrast RAG with traditional generative models.
Explore the future advancements and potential of RAG in NLP.


## 🛠️ What you will create
An analysis of traditional language model limitations.
A research report on the impact of attention in transformer models.
A functional RAG system for question answering.
An analytical report on BERT’s role in RAG.
A comparative analysis of RAG and traditional generative models.
A research summary on the future of RAG in NLP.

## Exercise 1: Identifying Limitations of Traditional Language Models
Objective: Recognize the challenges faced by traditional language models in handling long-range dependencies and complex contexts.

Instructions:

Consider the following sentence: “The scientist, who had been working on the project for years, finally made a breakthrough discovery.”
Analyze how a traditional language model, which processes text sequentially, might struggle to capture the relationship between “scientist” and “discovery” due to the intervening words.
Explain how this limitation can affect the model’s ability to accurately understand the sentence’s meaning and perform tasks like question answering or summarization.
Discuss how the attention mechanism addresses this challenge by allowing the model to focus on relevant words regardless of their position in the sentence.

Dans la phrase :
**« The scientist, who had been working on the project for years, finally made a breakthrough discovery. »**
un modèle traditionnel séquentiel (comme les RNN simples) lit les mots un par un. À cause de la longue sous-phrase entre **« scientist »** et **« discovery »**, il peut **oublier** ou **diluer** l'information pertinente du **« scientist »**, ce qui nuit à la compréhension.

#### Impact de cette limite :

* **Compréhension imprécise** : Le modèle peut mal relier le sujet principal à l'action clé.
* **Question-answering** : S'il est demandé *"Who made the discovery?"*, le modèle pourrait être confus.
* **Résumé** : Il pourrait donner un résumé erroné, en négligeant l'élément essentiel.

#### Solution par l'attention :

L'**attention** (mécanisme clé des Transformers) permet au modèle :

* de **pondérer l'importance de chaque mot**,
* de **se concentrer directement sur “scientist”**, même si des mots intermédiaires existent,
* d'améliorer **précision et cohérence** dans les tâches de compréhension et de génération.

L'attention supprime le problème de distance et permet une meilleure gestion des dépendances à long terme.


## Exercise 2: Exploring the Impact of Attention in Transformers
Objective: Understand how the attention mechanism enhances the capabilities of transformer models in various NLP tasks.

Instructions:

Choose an NLP task, such as machine translation, text summarization, or question answering.
Research how transformer models, like BERT or GPT, utilize the attention mechanism to achieve state-of-the-art results in the chosen task.
Provide specific examples of how attention helps the model capture long-range dependencies, resolve ambiguities, and handle complex contexts.
Compare the performance of transformer models with and without attention mechanisms on the chosen task, highlighting the improvements achieved through attention.

#### Tâche choisie : **Traduction automatique**

#### Utilisation de l’attention dans les Transformers :

Les modèles comme **GPT** et surtout **BERT** (via architectures encoder-decoders comme **BART** ou **T5**) utilisent **l'attention multi-têtes** pour :

* **aligner dynamiquement** chaque mot source avec les mots cibles,
* **capturer les dépendances longues** (par exemple, un verbe peut être influencé par un sujet très éloigné),
* **gérer les ambiguïtés lexicales** selon le contexte global.

#### Exemples spécifiques :

* Phrase anglaise : *“The bank approved the loan.”*

  * Sans attention : **“bank”** pourrait être traduit par “rivière”.
  * Avec attention : le contexte “approved the loan” oriente vers **“banque”**.
* Longues phrases complexes : les Transformers capturent la **structure grammaticale globale** et évitent les erreurs de segmentation ou de traduction littérale.

#### Comparaison avec et sans attention :

* **Sans attention** (RNN ou CNN) : erreurs fréquentes sur longues distances, incohérences grammaticales, pertes de contexte.
* **Avec attention** (Transformers) : traductions plus fluides, sens conservé, meilleure gestion de la syntaxe complexe.

#### Conclusion :

L'**attention** est la clé des performances **SOTA (State-of-the-Art)** en traduction automatique, permettant **précision**, **fluidité** et **compréhension contextuelle**.


## Exercise 3: Building a Simple RAG System for Question Answering
Objective: Implement a basic Retrieval-Augmented Generation (RAG) system for answering questions based on a given knowledge source.

Instructions:

Choose a knowledge source, such as a Wikipedia article or a collection of text documents.
Use a pre-trained BERT model to generate embeddings for the knowledge source and store them in a vector database (e.g., FAISS).
Implement a retriever that takes a user question as input, generates its embedding using BERT, and retrieves the most relevant documents from the vector database.
Use a pre-trained GPT model as the generator. Feed the retrieved documents and the user question to the generator to produce an answer.
Test the RAG system with various questions and evaluate its performance in terms of accuracy and relevance of the generated answers.

In [2]:
!pip install sentence-transformers transformers faiss-cpu



[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from sentence_transformers import SentenceTransformer
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import faiss
import numpy as np

# 1. Modèle d'encodage (Retriever)
retriever = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# 2. Knowledge base (source documentaire)
documents = [
    "Alan Turing est considéré comme le père de l'intelligence artificielle.",
    "L'intelligence artificielle est un domaine de l'informatique.",
    "Les réseaux de neurones sont largement utilisés dans l'IA moderne.",
    "Le test de Turing a été proposé pour évaluer l'intelligence des machines."
]

# 3. Encodage des documents
document_embeddings = retriever.encode(documents, convert_to_numpy=True)

# 4. Indexation FAISS
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(document_embeddings)

# 5. Modèle GPT (Generator)
gpt_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt_model = GPT2LMHeadModel.from_pretrained('gpt2')

# 6. Fonction question-answering RAG
def answer_question(question, top_k=2):
    # Encode la question
    question_embedding = retriever.encode([question], convert_to_numpy=True)
    
    # Recherche des documents les plus proches
    distances, indices = index.search(question_embedding, top_k)
    retrieved_docs = [documents[idx] for idx in indices[0]]

    # Construit le prompt
    prompt = "Contexte : " + " ".join(retrieved_docs) + "\nQuestion : " + question + "\nRéponse :"

    # Génération avec GPT
    input_ids = gpt_tokenizer.encode(prompt, return_tensors='pt')
    output_ids = gpt_model.generate(input_ids, max_length=100, num_return_sequences=1)
    answer = gpt_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return answer

# 7. Exemple de test
question = "Qui est le père de l'intelligence artificielle ?"
print(answer_question(question))


c:\Users\chume\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Contexte : Alan Turing est considéré comme le père de l'intelligence artificielle. L'intelligence artificielle est un domaine de l'informatique.
Question : Qui est le père de l'intelligence artificielle ?
Réponse : I am not sure if you can answer this question.
Question : I am not sure if you can answer this question.
Question : I am not sure if you can answer


## Exercise 4: Analyzing the Role of BERT in RAG Systems
Objective: Investigate how BERT contributes to the effectiveness of RAG systems in different applications.

Instructions:

Choose a specific application of RAG, such as question answering, fact-checking, or content creation.
Explain how BERT is used in the retrieval component of the RAG system to generate embeddings for documents and queries.
Discuss the advantages of using BERT embeddings for retrieval compared to other methods, such as TF-IDF or word embeddings.
Analyze how the quality of BERT embeddings affects the overall performance of the RAG system in the chosen application.

#### Application choisie : **Question Answering (QA)**

#### Rôle de BERT :

* Dans un système **RAG**, BERT sert de **retriever** :

  * Il encode les **documents** et les **questions** en vecteurs denses (**embeddings**).
  * La **similarité cosinus** entre embeddings permet d’identifier les documents pertinents.

#### Avantages par rapport à TF-IDF / Word2Vec :

* **TF-IDF** → basé sur la fréquence des mots, incapable de capturer le **sens global**.
* **Word2Vec** → embeddings fixes, peu sensibles au **contexte**.
* **BERT** → embeddings **contextualisés**, meilleure compréhension du **sens global** de la question et des documents, même avec synonymes ou reformulations.

#### Impact des embeddings BERT sur RAG :

* **Meilleure pertinence des documents récupérés**, donc **meilleures réponses générées**.
* **Réduction des erreurs factuelles**, car le générateur GPT produit sur une base d’informations **plus fiables**.
* Sans bons embeddings → documents hors sujet → **hallucinations GPT** aggravées.

#### Conclusion :

La **qualité des embeddings BERT est cruciale** : un bon retriever (BERT) = un RAG efficace, surtout en QA où la **pertinence du contexte** est primordiale.


## Exercise 5: Comparing RAG with Traditional Generative Models
Objective: Understand the advantages and limitations of RAG compared to traditional generative models like GPT.

Instructions:

Compare and contrast the architectures of RAG systems and traditional generative models.
Discuss the strengths and weaknesses of each approach in terms of accuracy, factual correctness, and ability to handle new information.
Provide examples of scenarios where RAG would be more suitable than traditional generative models, and vice versa.
Analyze the trade-offs between using RAG and traditional generative models in different applications.

| Critères                   | **RAG (Retriever + Generator)**                         | **Modèles génératifs traditionnels (GPT, LLM)**              |
| -------------------------- | ------------------------------------------------------- | ------------------------------------------------------------ |
| **Architecture**           | Combinaison : **BERT retriever** + **GPT générateur**   | Modèle génératif pur (GPT, Llama), pas de retrieval externe  |
| **Précision**              | +++ : récupère des infos précises, moins de dérives     | + : peut halluciner, généralisations parfois imprécises      |
| **Correction factuelle**   | +++ : supporte ses réponses sur des documents récupérés | - : hallucinations fréquentes, surtout hors domaine entraîné |
| **Accès à infos récentes** | +++ : mises à jour possibles via index                  | - : statique, nécessite fine-tuning ou re-training           |
| **Vitesse**                | - : deux étapes (retrieval + génération) = plus lent    | + : rapide, tout intégré dans un seul modèle                 |
| **Flexibilité créative**   | - : plus factuel, moins créatif                         | +++ : très créatif, utile pour rédaction libre               |

#### Exemples :

* **RAG préférable** :

  * **QA factuelle** : *“Quelle est la capitale de l’Australie ?”*
  * **Recherche documentaire**, **fact-checking**, **assistants spécialisés**.
* **GPT seul préférable** :

  * **Création de contenu créatif**, **brainstorming**, **rédaction libre** sans contraintes factuelles.

#### Analyse des compromis :

* **RAG = précision**, mais plus complexe et plus lent.
* **GPT seul = simplicité et rapidité**, mais plus de risques d’erreurs.
* Le choix dépend du besoin : **exactitude vs créativité**.


## Exercise 6: Exploring the Future of RAG in NLP
Objective: Research the latest advancements and potential future directions of RAG in natural language processing.

Instructions:

Investigate recent research papers and articles on RAG, focusing on new techniques, applications, and challenges.
Identify emerging trends in RAG, such as the use of different retrieval methods, knowledge sources, and generator architectures.
Discuss the potential impact of RAG on various NLP applications, such as conversational AI, information retrieval, and content generation.
Analyze the challenges and opportunities for future research in RAG, considering factors like scalability, efficiency, and ethical considerations.

##  Avancées récentes

* **Contexte suffisant pour la génération**
  Google Research a proposé en mai 2025 une métrique de *sufficient context* : elle évalue si les documents récupérés contiennent vraiment l’information nécessaire, ce qui aide à réduire les hallucinations ([research.google][1]).
* **Dynamiques et paramétriques**
  Le RAG évolue vers des pipelines **adaptatifs** : au lieu d’un simple « retrieve-then-generate », les systèmes *Dynamic RAG* choisissent **quand et quoi** récupérer, tandis que *Parametric RAG* injecte activement la connaissance directement dans les paramètres du modèle ([arXiv][2]).
* **Intégration causale & multimodale**

  * *CausalRAG* utilise des graphes causaux pour renforcer la cohérence dans les réponses ([arXiv][3]).
  * *MRAG* combine récupération et génération de données **texte + images/vidéos** pour enrichir la compréhension multimodale ([arXiv][4]).

---

##  Tendances émergentes

1. **Retrieval via RL**
   L’architecture *RAG-RL* améliore les capacités de reasoning et d’adaptation au cours de l'entraînement en renforcement ([LinkedIn][5]).
2. **Graphes de connaissances (GraphRAG)**
   Extraction de sous-graphes pour renforcer la structure des connaissances, au-delà des simples chunks textuels ([Wikipédia][6]).
3. **Recherche hybride & late interaction**
   Fusion de méthodes vectorielles et TF-IDF, et échanges plus fins entre tokens pour de meilleurs résultats .
4. **Multilinguisme & bas‑ressources**
   Modèles comme *NLLB‑E5* ouvrent des perspectives pour les langues peu représentées ([Medium][7]).

---

##  Applications potentielles

* **Santé** : diagnostic en temps réel intégré avec recherche des dernières publications médicales .
* **Agents conversationnels spécialisés** : respectant la sécurité et autonomie grâce à architectures hybrides agent‑based ([TechRadar][8]).
* **Analyse d’images + textes** : pour la modélisation multimodale, MRAG est prometteur.

---

##  Défis & enjeux

* **Scalabilité & latence** : plus de données = index lourds et lenteurs possibles ([harrisonclarke.com][9]).
* **Biais & qualité des données** : contenu erroné ou biaisé peut compromettre la qualité des réponses .
* **Éthique et sécurité** : risques d’exposition de données sensibles ou de contenus manipulables ([TechRadar][8]).
* **Évaluation** : nouveaux frameworks (Ares, RAG‑bench…) apparaissent, car les métriques classiques sont insuffisantes ([arXiv][10]).

---

##  Opportunités futures

* **RAG dynamique + paramétrique** pour plus d’efficience et adaptativité.
* **MRAG** pour des systèmes dotés d’une compréhension multimodale.
* **GraphRAG** et **CausalRAG** pour de la génération plus structurée et logique.
* **Agent‑based architectures** pour un RAG plus sécurisé et conforme ([arXiv][2], [Wikipédia][11], [TechRadar][8]).

---

###  En résumé

Le RAG n’est pas mort, mais en pleine **métamorphose**. Ses directions à suivre : adaptation dynamique, intégration multimodale, causalité, graphes, sécurité renforcée, et évaluation adaptée au contexte. Les domaines comme la santé, le juridique ou les agents spécialisés devraient bénéficier pleinement de ces avancées.

---


## Bilan et conclusion

### Les exercices ont montré :

* Les **limites des anciens modèles séquentiels** (oubli, dépendances longues).
* L'**importance du mécanisme d’attention** pour capter contexte et précision.
* Le **RAG** combine retrieval (BERT) + génération (GPT), améliorant la **précision** et réduisant les **hallucinations**.
* **BERT** est essentiel dans RAG pour récupérer des informations pertinentes, bien mieux que TF-IDF ou Word2Vec.
* **RAG** est **plus robuste** pour les tâches factuelles, mais **plus complexe** que GPT seul.
* L’avenir du RAG s’oriente vers des systèmes **dynamiques**, **multimodaux**, **orientés causalité** et **agents autonomes**.

---

###  **Conclusion rapide**

RAG est aujourd'hui la solution **la plus efficace pour combiner génération et factualité**. Les prochaines évolutions viseront des systèmes **plus intelligents, fiables et adaptés en temps réel**, particulièrement utiles pour les applications professionnelles exigeant **précision et contrôle**.
